In [38]:
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from openai import OpenAI


load_dotenv(override=True)
openai_client = OpenAI()

In [39]:
class translateTool(BaseModel):
    text: str = Field(..., description="The text to be translated.")
    source_language: str = Field(..., description="The source language of the text.")
    target_language: str = Field(..., description="The target language for the translation.")

@tool
def translate_text(text: str, source_language: str, target_language: str) -> str:
    """
    Translates text from source_language to target_language using OpenAI.
    """
    prompt = (
        f"Translate the following text from {source_language} into {target_language}. "
        "Respond with only the translated text.\n\n"
        f"{text}"
    )
    

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt,
        max_output_tokens=1024,
    )

    return response.output_text.strip()
agent=create_agent(model="openai:gpt-5.4-mini",system_prompt="You are a helpfil assistant who answers concisely",checkpointer=MemorySaver(),response_format=translateTool,tools=[translate_text])
config={"configurable":{"thread_id":"translator"}}
result= agent.invoke({"messages":[{"role":"user","content":"Hello, can you help me with translating a sentence from English to German?"}]}, config=config)
print(result["messages"][-1].content)

{"text":"Hello, can you help me with translating a sentence from English to German?","source_language":"English","target_language":"German"}


In [44]:
result= await agent.ainvoke({"messages":[{"role":"user","content":"Hello, can you help me with translating a sentence from English to German?"}]},config=config)
print(result["messages"][-1].content)

{"text":"Hallo, kannst du mir helfen, einen Satz von Englisch ins Deutsche zu übersetzen?","source_language":"English","target_language":"German"}


In [41]:
result= agent.invoke({"messages":[{"role":"user","content":"Translate 'Hello, how are you?'."}]}, config=config)
print(result["messages"][-1].content)

{"text":"Hello, how are you?","source_language":"English","target_language":"German"}


In [42]:


agent_with_tool = agent.invoke({"messages":[{"role":"user","content":"Translate 'Hello, how are you?' from English to German."}]}, config=config)

print(agent_with_tool["messages"][-1].content)
print(agent_with_tool["structured_response"])


{"text":"Hallo, wie geht es dir?","source_language":"English","target_language":"German"}
text='Hallo, wie geht es dir?' source_language='English' target_language='German'


In [43]:
import gradio as gr
def chat(user_input:str,history):
    result= agent.invoke({"messages":[{"role":"user","content":user_input}]}, config=config)
   #history.append((user_input,result["messages"][-1].content))
    return f" {result['messages'][-1].content}"

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
